In [1]:
# ============================================================
# Step 1 ONLY: split into 3 dataframes (odd_one_out / exploration / memory)
# IMPORTANT per your request:
#   - Exclude ONLY: stim72_mapping.csv
#   - Do NOT write any output CSVs yet
#   - Robust to the newer Ivy CSV structure (including malformed survey rows)
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import csv

# ----------------------------
# Load ALL CSVs in notebook folder (exclude only stim72_mapping.csv)
# ----------------------------

# NOTE:
# Do NOT replace the flexible reader below with plain pd.read_csv(p).
# data_ivy.csv contains a few malformed over-wide rows, and plain pandas parsing will fail.
try:
    import ipynbname  # pip install ipynbname
    base_dir = Path(ipynbname.path()).parent
except Exception:
    base_dir = Path.cwd()

print("Using folder:", base_dir)

EXCLUDE_FILES = {"stim72_mapping.csv"}  # ONLY exclude this one

csv_paths = sorted([p for p in base_dir.glob("*.csv") if p.name not in EXCLUDE_FILES])
if not csv_paths:
    raise FileNotFoundError(f"No .csv files found in: {base_dir}")

print(f"Found {len(csv_paths)} CSV(s).")
maybe_outputs = {"all_with_phase.csv", "odd_one_out.csv", "exploration.csv", "memory.csv"}
present_outputs = sorted(set(p.name for p in csv_paths).intersection(maybe_outputs))
if present_outputs:
    print("WARNING: These CSVs are included and may duplicate rows if they are generated outputs:")
    print("  ", present_outputs)

def read_csv_flexible(path: Path) -> pd.DataFrame:
    """
    Robust loader for the newer task exports.

    Why this is needed:
      - data_ivy.csv can contain a few rows with more fields than the header
        because of later survey exports / embedded commas.
      - We are ignoring survey sections here anyway, so it is safer to
        normalize row width than to fail on read.
    """
    with open(path, newline="", encoding="utf-8", errors="replace") as f:
        reader = csv.reader(f)
        rows = list(reader)

    if not rows:
        return pd.DataFrame()

    header = rows[0]
    n_cols = len(header)

    fixed_rows = []
    n_short = 0
    n_long = 0

    for row in rows[1:]:
        if len(row) < n_cols:
            row = row + [""] * (n_cols - len(row))
            n_short += 1
        elif len(row) > n_cols:
            row = row[:n_cols]
            n_long += 1
        fixed_rows.append(row)

    df = pd.DataFrame(fixed_rows, columns=header).replace("", pd.NA).convert_dtypes()

    if n_short or n_long:
        print(f"  {path.name}: normalized {n_short} short row(s), truncated {n_long} long row(s).")

    return df

dfs = []
for p in csv_paths:
    d = read_csv_flexible(p)
    d["__source_file__"] = p.name
    d["__row_in_file__"] = np.arange(len(d))
    dfs.append(d)

df_all = pd.concat(dfs, ignore_index=True, sort=False)
print("Combined df_all shape:", df_all.shape)

# ----------------------------
# Phase splitting logic
# ----------------------------
if "trial_type" not in df_all.columns:
    raise KeyError("Expected a 'trial_type' column to split phases, but it was not found.")

PARTICIPANT_COL = "id" if "id" in df_all.columns else "__source_file__"

# `explore_seen` is not part of the exported exploration phase.
# Keep it separate and use it only for memory seen/unseen classification.
EXPLORE_TYPES = {"explore_decision", "room_choice"}
MEMORY_TYPES  = {"memory_choice", "oldnew_response"}
OOO_TYPE      = "odd_one_out"

print("\nUnique trial_type values (top counts):")
print(df_all["trial_type"].value_counts(dropna=False).head(30))

def _assign_phases_one_participant(g: pd.DataFrame) -> pd.DataFrame:
    g = g.copy()

    if "time_elapsed" in g.columns:
        g["__time_elapsed_num__"] = pd.to_numeric(g["time_elapsed"], errors="coerce")
        g = g.sort_values(["__time_elapsed_num__", "__row_in_file__"], kind="mergesort", na_position="last")
    else:
        g = g.sort_values(["__row_in_file__"], kind="mergesort")

    g["__pos__"] = np.arange(len(g))

    explore_pos = g.loc[g["trial_type"].isin(EXPLORE_TYPES), "__pos__"]
    memory_pos  = g.loc[g["trial_type"].isin(MEMORY_TYPES), "__pos__"]

    explore_start = int(explore_pos.min()) if len(explore_pos) else np.inf
    memory_end    = int(memory_pos.max()) if len(memory_pos) else -np.inf

    g["phase"] = np.where(g["trial_type"].isin(EXPLORE_TYPES), "exploration",
                 np.where(g["trial_type"].isin(MEMORY_TYPES), "memory",
                 np.where(g["trial_type"].eq(OOO_TYPE), "odd_one_out",
                          "other")))

    g["ooo_block"] = pd.Series(pd.NA, index=g.index, dtype="Int64")
    is_ooo = g["trial_type"].eq(OOO_TYPE)
    g.loc[is_ooo & (g["__pos__"] < explore_start), "ooo_block"] = 1
    g.loc[is_ooo & (g["__pos__"] > memory_end), "ooo_block"] = 2

    g["memory_subphase"] = pd.Series(pd.NA, index=g.index, dtype="string")
    g.loc[g["trial_type"].eq("memory_choice"), "memory_subphase"] = "value_judgment"
    g.loc[g["trial_type"].eq("oldnew_response"), "memory_subphase"] = "oldnew_judgment"

    return g.drop(columns=["__pos__", "__time_elapsed_num__"], errors="ignore")

df_tagged = df_all.groupby(PARTICIPANT_COL, group_keys=False).apply(_assign_phases_one_participant)

df_odd  = df_tagged[df_tagged["phase"] == "odd_one_out"].copy()
df_expl = df_tagged[df_tagged["phase"] == "exploration"].copy()
df_mem  = df_tagged[df_tagged["phase"] == "memory"].copy()
df_explore_seen = df_tagged[df_tagged["trial_type"].eq("explore_seen")].copy()

Using folder: /Users/jianleguo/Desktop/Mario_Pilot_Data/participant_csv
Found 6 CSV(s).
  data_5d4c78071b920f00198e8533.csv: normalized 0 short row(s), truncated 4 long row(s).
  data_66ba32b2a868595bbef88468.csv: normalized 0 short row(s), truncated 4 long row(s).
  data_695ff63580d6ce826e06277f.csv: normalized 0 short row(s), truncated 4 long row(s).
  data_697ed59824964096e537d7d3.csv: normalized 0 short row(s), truncated 5 long row(s).
  data_69aa333e1f57c2124c5706ad.csv: normalized 0 short row(s), truncated 5 long row(s).
  data_69c312e7e225af97496fb371.csv: normalized 0 short row(s), truncated 4 long row(s).
Combined df_all shape: (8377, 395)

Unique trial_type values (top counts):
trial_type
explore_decision              3172
explore_seen                  3063
odd_one_out                   1116
oldnew_response                368
memory_choice                  361
room_choice                    180
post_survey_builder             48
color_blind_survey_plate        24
post_survey 

In [2]:
# ---- (A) OOO: label phase in trial_index ----
df_odd = df_odd.copy()

# keep original trial_index if present
if "trial_index" in df_odd.columns:
    df_odd["trial_index_within_ooo"] = df_odd["trial_index"]

# trial_index becomes OOO_1 / OOO_2 (based on your ooo_block)
df_odd["trial_index"] = (
    df_odd["ooo_block"]
      .map({1: "OOO_1", 2: "OOO_2"})
      .fillna("OOO_unknown")   # in case something falls between explore and memory
      .astype("string")
)

# optional: keep an explicit column too (sometimes nicer than overloading trial_index)
df_odd["ooo_phase"] = df_odd["trial_index"]


# ---- (B) drop all-NA columns in each df ----
def drop_all_na_cols(d: pd.DataFrame) -> pd.DataFrame:
    return d.dropna(axis=1, how="all")

df_odd  = drop_all_na_cols(df_odd)
df_expl = drop_all_na_cols(df_expl)
df_mem  = drop_all_na_cols(df_mem)
df_explore_seen = drop_all_na_cols(df_explore_seen)


In [3]:
import numpy as np
import pandas as pd
from pathlib import Path

# -----------------------
# Settings (edit if needed)
# -----------------------
PARTICIPANT_COL = "id" if "id" in df_mem.columns else "__source_file__"

# response codes
OLD_CODE = 1
NEW_CODE = 2


# -----------------------
# Helpers
# -----------------------
def make_pid_key(s: pd.Series) -> pd.Series:
    """
    Stable participant key to avoid mixed-type sorting/grouping:
      - 1 or 1.0 -> "1"
      - strings stay strings
      - missing stays NA
    """
    s0 = s.copy()

    out = s0.astype("string")
    num = pd.to_numeric(s0, errors="coerce")

    # Convert to a plain float64 numpy array so numpy ops do not choke on
    # pandas extension dtypes (for example nullable Float64 / string-backed data).
    num_np = num.to_numpy(dtype="float64", na_value=np.nan)
    intlike = pd.Series(
        np.isfinite(num_np) & np.isclose(np.mod(num_np, 1.0), 0.0),
        index=s0.index
    )

    out.loc[intlike] = num.loc[intlike].astype("Int64").astype("string")
    out = out.replace("nan", pd.NA)
    return out

def _basename(x):
    if pd.isna(x):
        return np.nan
    return Path(str(x)).name  # works for full paths or filenames

def _get_seen_set(expl_df, explore_seen_df=None):
    """
    Build the per-participant seen-image set used for memory old/new scoring.

    Old data used exploration rows with `stimulus`.
    Ivy adds separate `explore_seen` rows with `imagefilename`.
    We keep those rows out of the exported exploration phase and use them only here.
    """
    seen = set()

    if expl_df is not None:
        for col in ["stimulus"]:
            if col in expl_df.columns:
                seen |= set(expl_df[col].dropna().map(_basename))

    if explore_seen_df is not None:
        for col in ["imagefilename"]:
            if col in explore_seen_df.columns:
                seen |= set(explore_seen_df[col].dropna().map(_basename))

    if seen:
        return seen

    fallback_frames = [d for d in [expl_df, explore_seen_df] if d is not None]
    png_series = []
    for d in fallback_frames:
        png_cols = [c for c in d.columns if d[c].astype(str).str.contains(r"\.png", na=False).any()]
        png_series.extend([d[c].dropna().map(_basename) for c in png_cols])

    if not png_series:
        return set()

    return set(pd.concat(png_series).unique())

def _valid_oldnew_rows(mem_df):
    if "trial_type" in mem_df.columns:
        m = mem_df["trial_type"].eq("oldnew_response")
    else:
        m = mem_df.get("memory_subphase", pd.Series(False, index=mem_df.index)).eq("oldnew_judgment")

    out = mem_df.loc[m].copy()

    if "event" in out.columns:
        out = out[out["event"].isna()]

    if "tested_mushroom_image" not in out.columns:
        raise KeyError("df_mem is missing 'tested_mushroom_image' needed for old/new scoring.")
    if "response" not in out.columns:
        raise KeyError("df_mem is missing 'response' needed for old/new scoring.")

    out = out[out["tested_mushroom_image"].notna() & out["response"].notna()].copy()
    return out


# -----------------------
# Make participant key (_pid) to avoid mixed-type issues
# -----------------------
df_mem = df_mem.copy()
df_expl = df_expl.copy()
df_explore_seen = df_explore_seen.copy()

df_mem["_pid"] = make_pid_key(df_mem[PARTICIPANT_COL])

if PARTICIPANT_COL in df_expl.columns:
    df_expl["_pid"] = make_pid_key(df_expl[PARTICIPANT_COL])
else:
    df_expl["_pid"] = pd.NA

if PARTICIPANT_COL in df_explore_seen.columns:
    df_explore_seen["_pid"] = make_pid_key(df_explore_seen[PARTICIPANT_COL])
else:
    df_explore_seen["_pid"] = pd.NA

if "__source_file__" in df_mem.columns:
    df_mem["_pid"] = df_mem["_pid"].fillna(df_mem["__source_file__"].astype("string"))
if "__source_file__" in df_expl.columns:
    df_expl["_pid"] = df_expl["_pid"].fillna(df_expl["__source_file__"].astype("string"))
if "__source_file__" in df_explore_seen.columns:
    df_explore_seen["_pid"] = df_explore_seen["_pid"].fillna(df_explore_seen["__source_file__"].astype("string"))


# -----------------------
# Compute per-participant metrics
# -----------------------
rows = []

for pid, g_mem in df_mem.groupby("_pid", dropna=False):
    g_expl = df_expl[df_expl["_pid"] == pid] if "_pid" in df_expl.columns else df_expl
    g_explore_seen = df_explore_seen[df_explore_seen["_pid"] == pid] if "_pid" in df_explore_seen.columns else df_explore_seen

    seen_set = _get_seen_set(g_expl, g_explore_seen)

    oldnew = _valid_oldnew_rows(g_mem)
    if oldnew.empty:
        rows.append({
            "_pid": pid,
            "n_valid": 0,
            "n_seen": 0,
            "n_unseen": 0,
            "accuracy": np.nan,
            "p_old_given_seen": np.nan,
            "p_old_given_unseen": np.nan,
            "bias_index_hit_minus_fa": np.nan,
        })
        continue

    oldnew["tested_base"] = oldnew["tested_mushroom_image"].map(_basename)
    oldnew["seen"] = oldnew["tested_base"].isin(seen_set)

    resp = pd.to_numeric(oldnew["response"], errors="coerce")
    oldnew["resp_old"] = resp.eq(OLD_CODE)
    oldnew["resp_new"] = resp.eq(NEW_CODE)

    oldnew_valid = oldnew[oldnew["resp_old"] | oldnew["resp_new"]].copy()

    if oldnew_valid.empty:
        rows.append({
            "_pid": pid,
            "n_valid": 0,
            "n_seen": 0,
            "n_unseen": 0,
            "accuracy": np.nan,
            "p_old_given_seen": np.nan,
            "p_old_given_unseen": np.nan,
            "bias_index_hit_minus_fa": np.nan,
        })
        continue

    correct = (oldnew_valid["seen"] & oldnew_valid["resp_old"]) | (~oldnew_valid["seen"] & oldnew_valid["resp_new"])
    acc = correct.mean()

    seen_rows = oldnew_valid[oldnew_valid["seen"]]
    unseen_rows = oldnew_valid[~oldnew_valid["seen"]]

    p_old_seen = seen_rows["resp_old"].mean() if len(seen_rows) else np.nan
    p_old_unseen = unseen_rows["resp_old"].mean() if len(unseen_rows) else np.nan

    bias_index = p_old_seen - p_old_unseen if (pd.notna(p_old_seen) and pd.notna(p_old_unseen)) else np.nan

    rows.append({
        "_pid": pid,
        "n_valid": int(len(oldnew_valid)),
        "n_seen": int(seen_rows.shape[0]),
        "n_unseen": int(unseen_rows.shape[0]),
        "accuracy": float(acc),
        "p_old_given_seen": float(p_old_seen) if pd.notna(p_old_seen) else np.nan,
        "p_old_given_unseen": float(p_old_unseen) if pd.notna(p_old_unseen) else np.nan,
        "bias_index_hit_minus_fa": float(bias_index) if pd.notna(bias_index) else np.nan,
    })

summary = pd.DataFrame(rows)


# -----------------------
# 1) Print each participant accuracy + average accuracy
# -----------------------
print("Per-participant old/new performance (using exploration + explore_seen rows to define seen):")
display(summary.sort_values("_pid"))

print("\nAverage accuracy across participants:")
print(summary["accuracy"].mean(skipna=True))


# -----------------------
# 2) Print bias-corrected index: P(old|seen) - P(old|unseen)
# -----------------------
print("\nPer-participant bias index (P(old|seen) - P(old|unseen)):")
print(summary[["_pid", "bias_index_hit_minus_fa"]].sort_values("_pid").to_string(index=False))

print("\nAverage bias index across participants:")
print(summary["bias_index_hit_minus_fa"].mean(skipna=True))


# -----------------------
# Optional sanity check: response code distribution in old/new trials
# -----------------------
print("\nSanity check: response code counts in old/new trials (valid rows only):")
oldnew_all = _valid_oldnew_rows(df_mem)
print(pd.to_numeric(oldnew_all["response"], errors="coerce").value_counts(dropna=False).head(20))

Per-participant old/new performance (using exploration + explore_seen rows to define seen):


,_pid,n_valid,n_seen,n_unseen,accuracy,p_old_given_seen,p_old_given_unseen,bias_index_hit_minus_fa
0,5d4c78071b920f00198e8533,60,24,36,0.516667,0.791667,0.666667,0.125000
1,66ba32b2a868595bbef88468,60,12,48,0.783333,0.166667,0.062500,0.104167
2,695ff63580d6ce826e06277f,60,14,46,0.600000,0.428571,0.347826,0.080745
3,697ed59824964096e537d7d3,60,23,37,0.566667,0.086957,0.135135,-0.048179
4,69aa333e1f57c2124c5706ad,60,15,45,0.583333,0.133333,0.266667,-0.133333
5,69c312e7e225af97496fb371,60,16,44,0.533333,0.250000,0.363636,-0.113636



Average accuracy across participants:
0.5972222222222222

Per-participant bias index (P(old|seen) - P(old|unseen)):
                    _pid  bias_index_hit_minus_fa
5d4c78071b920f00198e8533                 0.125000
66ba32b2a868595bbef88468                 0.104167
695ff63580d6ce826e06277f                 0.080745
697ed59824964096e537d7d3                -0.048179
69aa333e1f57c2124c5706ad                -0.133333
69c312e7e225af97496fb371                -0.113636

Average bias index across participants:
0.0024606163193119693

Sanity check: response code counts in old/new trials (valid rows only):
response
2       249
1       111
<NA>      0
Name: count, dtype: Int64


In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================================================
# 0) Load stim72_mapping ONLY (df_odd/df_expl/df_mem already exist)
# =========================================================
MAP_PATHS = ["stim72_mapping.csv", "/mnt/data/stim72_mapping.csv"]
map_path = next((p for p in MAP_PATHS if Path(p).exists()), None)
if map_path is None:
    raise FileNotFoundError("Could not find stim72_mapping.csv (checked local + /mnt/data).")

stim72_map = pd.read_csv(map_path)[["stim72_id", "key"]]
key2id = dict(zip(stim72_map["key"], stim72_map["stim72_id"]))

# =========================================================
# 1) Core helpers (shared by ODD/EXPL/MEM)
# =========================================================
def _norm_fname(s: pd.Series) -> pd.Series:
    return (s.astype(str)
             .str.replace("\\", "/", regex=False)
             .str.rsplit("/", n=1).str[-1]
             .str.strip())

def _extract_color(img_series: pd.Series) -> pd.Series:
    base = _norm_fname(img_series)
    out = base.str.split("-", n=1).str[0].str.lower()
    out = out.where(~base.isin(["<NA>", "nan", "None"]), pd.NA)
    return out.astype("string")

def _parse_color_stem_cap(img_series: pd.Series):
    parts = (_norm_fname(img_series)
             .str.replace(".png", "", regex=False)
             .str.split("-", expand=True))

    color = parts[0].str.lower()
    stem  = pd.to_numeric(parts[1], errors="coerce")
    cap   = pd.to_numeric(parts[2], errors="coerce")

    stem_type = np.where(stem < 6, "thin", np.where(stem > 10, "thick", "neutral"))
    cap_type  = np.where(cap  < 0.8, "flat", np.where(cap  > 1.4, "round", "neutral"))
    return color, stem, cap, stem_type, cap_type

def add_stim72(df: pd.DataFrame, img_col: str, prefix: str) -> pd.DataFrame:
    """Adds: {prefix}_stim72_id, {prefix}_stem_type, {prefix}_cap_type (+ numeric stem/cap)."""
    if img_col not in df.columns:
        return df

    color, stem, cap, stem_type, cap_type = _parse_color_stem_cap(df[img_col])

    df[f"{prefix}_stem_type"] = stem_type
    df[f"{prefix}_cap_type"]  = cap_type
    df[f"{prefix}_stem_size"] = stem.astype("Int64")
    df[f"{prefix}_cap_round"] = cap

    key = color + "|" + stem_type + "|" + cap_type
    df[f"{prefix}_stim72_id"] = pd.Series(key.map(key2id), index=df.index).astype("Int64")
    return df

def label_chosen_from_three(df: pd.DataFrame,
                            stim_cols=("stimulus_1","stimulus_2","stimulus_3"),
                            chosen_col="chosen_image",
                            response_col="response") -> pd.DataFrame:
    s1, s2, s3 = stim_cols
    need = [s1, s2, s3]
    if not all(c in df.columns for c in need):
        return df

    df = add_stim72(df, s1, "stim1")
    df = add_stim72(df, s2, "stim2")
    df = add_stim72(df, s3, "stim3")

    c1, c2, c3 = _norm_fname(df[s1]), _norm_fname(df[s2]), _norm_fname(df[s3])
    if chosen_col in df.columns:
        ch = _norm_fname(df[chosen_col])
        opt_from_img = pd.Series(pd.NA, index=df.index, dtype="Int64")
        opt_from_img.loc[ch.eq(c1)] = 1
        opt_from_img.loc[ch.eq(c2)] = 2
        opt_from_img.loc[ch.eq(c3)] = 3
    else:
        opt_from_img = pd.Series(np.nan, index=df.index)

    opt_from_resp = pd.to_numeric(df.get(response_col), errors="coerce")
    df["chosen_option_123"] = opt_from_img.fillna(opt_from_resp).astype("Int64")

    df["chosen_stim72_id"] = pd.Series(pd.NA, index=df.index, dtype="Int64")
    df["chosen_stem_type"] = pd.Series(pd.NA, index=df.index, dtype="string")
    df["chosen_cap_type"]  = pd.Series(pd.NA, index=df.index, dtype="string")

    m1, m2, m3 = df["chosen_option_123"].eq(1), df["chosen_option_123"].eq(2), df["chosen_option_123"].eq(3)

    df.loc[m1, ["chosen_stim72_id", "chosen_stem_type", "chosen_cap_type"]] = \
        df.loc[m1, ["stim1_stim72_id", "stim1_stem_type", "stim1_cap_type"]].to_numpy()
    df.loc[m2, ["chosen_stim72_id", "chosen_stem_type", "chosen_cap_type"]] = \
        df.loc[m2, ["stim2_stim72_id", "stim2_stem_type", "stim2_cap_type"]].to_numpy()
    df.loc[m3, ["chosen_stim72_id", "chosen_stem_type", "chosen_cap_type"]] = \
        df.loc[m3, ["stim3_stim72_id", "stim3_stem_type", "stim3_cap_type"]].to_numpy()

    df["chosen_cap_props"] = "stem=" + df["chosen_stem_type"].astype(str) + "|cap=" + df["chosen_cap_type"].astype(str)
    return df

def label_selected_lr(df: pd.DataFrame,
                      left_col="left_mushroom_image",
                      right_col="right_mushroom_image",
                      selected_col="selected_mushroom_image",
                      response_col="response") -> pd.DataFrame:
    """
    For MEM (2AFC): adds left/right stim72 + selected stim72 + chosen_side ('left'/'right') + chosen_* labels.
    chosen_side uses selected image match; falls back to response (1=left, 2=right) if needed.
    """
    df = add_stim72(df, left_col, "left")
    df = add_stim72(df, right_col, "right")
    df = add_stim72(df, selected_col, "selected")
    df = add_stim72(df, "other_mushroom_image", "other")
    df = add_stim72(df, "tested_mushroom_image", "tested")

    if left_col in df.columns and right_col in df.columns:
        L = _norm_fname(df[left_col])
        R = _norm_fname(df[right_col])

        if selected_col in df.columns:
            S = _norm_fname(df[selected_col])
            side_from_img = pd.Series(pd.NA, index=df.index, dtype="string")
            side_from_img.loc[S.eq(L)] = "left"
            side_from_img.loc[S.eq(R)] = "right"
        else:
            side_from_img = pd.Series(np.nan, index=df.index)

        resp = pd.to_numeric(df.get(response_col), errors="coerce")
        side_from_resp = pd.Series(pd.NA, index=df.index, dtype="string")
        side_from_resp.loc[resp.eq(1)] = "left"
        side_from_resp.loc[resp.eq(2)] = "right"

        # keep original selected_side if already present; otherwise infer it
        if "selected_side" in df.columns:
            selected_side_existing = df["selected_side"].astype("string")
            df["chosen_side"] = selected_side_existing.fillna(side_from_img).fillna(side_from_resp)
        else:
            df["chosen_side"] = side_from_img.fillna(side_from_resp)

        df["chosen_stim72_id"] = pd.Series(pd.NA, index=df.index, dtype="Int64")
        df["chosen_stem_type"] = pd.Series(pd.NA, index=df.index, dtype="string")
        df["chosen_cap_type"]  = pd.Series(pd.NA, index=df.index, dtype="string")

        ml = df["chosen_side"].eq("left")
        mr = df["chosen_side"].eq("right")

        if "left_stim72_id" in df.columns:
            df.loc[ml, ["chosen_stim72_id", "chosen_stem_type", "chosen_cap_type"]] = \
                df.loc[ml, ["left_stim72_id", "left_stem_type", "left_cap_type"]].to_numpy()
        if "right_stim72_id" in df.columns:
            df.loc[mr, ["chosen_stim72_id", "chosen_stem_type", "chosen_cap_type"]] = \
                df.loc[mr, ["right_stim72_id", "right_stem_type", "right_cap_type"]].to_numpy()

        df["chosen_cap_props"] = "stem=" + df["chosen_stem_type"].astype(str) + "|cap=" + df["chosen_cap_type"].astype(str)

    return df

def _first_nonnull(s: pd.Series):
    s = s.dropna()
    return s.iloc[0] if len(s) else pd.NA

def _normalize_seen_status(s: pd.Series) -> pd.Series:
    out = s.astype("string").str.lower()
    norm = pd.Series(pd.NA, index=s.index, dtype="string")
    norm.loc[out.str.startswith("seen", na=False)] = "seen"
    norm.loc[out.str.startswith("unseen", na=False)] = "unseen"
    return norm

def _coalesce_series(df: pd.DataFrame, cols, dtype=None):
    out = pd.Series(pd.NA, index=df.index)
    for col in cols:
        if col in df.columns:
            out = out.fillna(df[col])
    if dtype == "Int64":
        return pd.to_numeric(out, errors="coerce").astype("Int64")
    if dtype == "string":
        return out.astype("string")
    return out

def enrich_memory_new_structure(df: pd.DataFrame) -> pd.DataFrame:
    """
    Keep all old columns as-is, and add new memory-level cleaned columns for the Ivy structure.

    Added columns:
      - memory_trial_key
      - lure_bin_trial / difficulty_bin_trial / difficulty_label_trial
      - separation_mode_trial / cross_side_trial / seen_type_trial / unseen_type_trial
      - memory_lure_bin
      - memory_difficulty_bin
      - memory_difficulty_label
      - memory_color_pair_type        ('within_color' / 'cross_color')
      - memory_seen_pair_type         ('seen_vs_seen' / 'seen_vs_unseen' / 'unseen_vs_unseen')
    """
    df = df.copy()

    if "trial_index" not in df.columns:
        return df

    participant_col = "id" if "id" in df.columns else "__source_file__"
    pid = df[participant_col].astype("string")
    if "__source_file__" in df.columns:
        pid = pid.fillna(df["__source_file__"].astype("string"))

    trial_index_str = df["trial_index"].astype("string")
    is_mem = df["trial_type"].isin(["memory_choice", "oldnew_response"]) if "trial_type" in df.columns else pd.Series(True, index=df.index)

    df["memory_trial_key"] = pd.Series(pd.NA, index=df.index, dtype="string")
    df.loc[is_mem, "memory_trial_key"] = pid.loc[is_mem] + "|" + trial_index_str.loc[is_mem]

    trial_fill_cols = [
        "lure_bin",
        "difficulty_bin",
        "difficulty_label",
        "separation_mode",
        "cross_side",
        "seen_type",
        "unseen_type",
        "left_mushroom_image",
        "right_mushroom_image",
        "left_mushroom_status",
        "right_mushroom_status",
        "selected_mushroom_status",
        "other_mushroom_status",
        "left_mushroom_lure_bin",
        "right_mushroom_lure_bin",
        "selected_mushroom_lure_bin",
        "other_mushroom_lure_bin",
        "tested_mushroom_lure_bin",
    ]

    for col in trial_fill_cols:
        if col in df.columns:
            df[f"{col}_trial"] = (
                df.groupby("memory_trial_key", dropna=False)[col]
                  .transform(_first_nonnull)
            )

    df["memory_lure_bin"] = _coalesce_series(
        df,
        [
            "lure_bin_trial",
            "tested_mushroom_lure_bin_trial",
            "selected_mushroom_lure_bin_trial",
            "left_mushroom_lure_bin_trial",
            "right_mushroom_lure_bin_trial",
            "other_mushroom_lure_bin_trial",
        ],
        dtype="Int64"
    )

    df["memory_difficulty_bin"] = _coalesce_series(df, ["difficulty_bin_trial"], dtype="Int64")
    df["memory_difficulty_label"] = _coalesce_series(df, ["difficulty_label_trial"], dtype="string")

    left_img_trial = _coalesce_series(df, ["left_mushroom_image_trial"], dtype="string")
    right_img_trial = _coalesce_series(df, ["right_mushroom_image_trial"], dtype="string")
    left_color = _extract_color(left_img_trial)
    right_color = _extract_color(right_img_trial)

    df["memory_color_pair_type"] = pd.Series(pd.NA, index=df.index, dtype="string")
    both_color = left_color.notna() & right_color.notna()
    df.loc[both_color & left_color.eq(right_color), "memory_color_pair_type"] = "within_color"
    df.loc[both_color & ~left_color.eq(right_color), "memory_color_pair_type"] = "cross_color"

    left_seen = _normalize_seen_status(_coalesce_series(df, ["left_mushroom_status_trial"], dtype="string"))
    right_seen = _normalize_seen_status(_coalesce_series(df, ["right_mushroom_status_trial"], dtype="string"))

    df["memory_seen_pair_type"] = pd.Series(pd.NA, index=df.index, dtype="string")
    df.loc[left_seen.eq("seen") & right_seen.eq("seen"), "memory_seen_pair_type"] = "seen_vs_seen"
    df.loc[left_seen.eq("unseen") & right_seen.eq("unseen"), "memory_seen_pair_type"] = "unseen_vs_unseen"
    df.loc[
        ((left_seen.eq("seen") & right_seen.eq("unseen")) |
         (left_seen.eq("unseen") & right_seen.eq("seen"))),
        "memory_seen_pair_type"
    ] = "seen_vs_unseen"

    return df

# =========================================================
# 2) Run on your three dataframes
# =========================================================
# --- df_odd (OOO / odd-one-out) ---
df_odd = label_chosen_from_three(
    df_odd,
    stim_cols=("stimulus_1", "stimulus_2", "stimulus_3"),
    chosen_col="chosen_image",
    response_col="response"
)

# --- df_expl (exploration) ---
df_expl = add_stim72(df_expl, "stimulus", "stimulus")
df_expl = add_stim72(df_expl, "imagefilename", "imagefilename")

# --- df_mem (memory 2AFC; left/right + selected + new Ivy fields) ---
df_mem = label_selected_lr(
    df_mem,
    left_col="left_mushroom_image",
    right_col="right_mushroom_image",
    selected_col="selected_mushroom_image",
    response_col="response"
)

df_mem = enrich_memory_new_structure(df_mem)

print("df_odd added:", [c for c in df_odd.columns if "stim72" in c or c.startswith("chosen_")][-12:])
print("df_expl added:", [c for c in df_expl.columns if "stim72" in c or c.startswith(("stimulus_", "imagefilename_"))][-14:])
print("df_mem new structure cols:", [c for c in df_mem.columns if c.startswith("memory_") or c.endswith("_trial")][-20:])

df_odd added: ['chosen_image', 'stim1_stim72_id', 'stim2_stim72_id', 'stim3_stim72_id', 'chosen_option_123', 'chosen_stim72_id', 'chosen_stem_type', 'chosen_cap_type', 'chosen_cap_props']
df_expl added: ['stimulus_stem_type', 'stimulus_cap_type', 'stimulus_stem_size', 'stimulus_cap_round', 'stimulus_stim72_id']
df_mem new structure cols: ['separation_mode_trial', 'cross_side_trial', 'seen_type_trial', 'unseen_type_trial', 'left_mushroom_image_trial', 'right_mushroom_image_trial', 'left_mushroom_status_trial', 'right_mushroom_status_trial', 'selected_mushroom_status_trial', 'other_mushroom_status_trial', 'left_mushroom_lure_bin_trial', 'right_mushroom_lure_bin_trial', 'selected_mushroom_lure_bin_trial', 'other_mushroom_lure_bin_trial', 'tested_mushroom_lure_bin_trial', 'memory_lure_bin', 'memory_difficulty_bin', 'memory_difficulty_label', 'memory_color_pair_type', 'memory_seen_pair_type']


In [5]:
from pathlib import Path

# folder name (created if it doesn't exist)
OUT_FOLDER = "cleaned_outputs"

try:
    import ipynbname
    base_dir = Path(ipynbname.path()).parent
except Exception:
    base_dir = Path.cwd()

out_dir = base_dir / OUT_FOLDER
out_dir.mkdir(parents=True, exist_ok=True)

df_expl.to_csv(out_dir / "cleaned_exploration_phase_data.csv", index=False)
df_odd.to_csv(out_dir / "cleaned_odd_one_out_data.csv", index=False)
df_mem.to_csv(out_dir / "cleaned_memory_phase_data.csv", index=False)

print("Saved to:", out_dir)


Saved to: /Users/jianleguo/Desktop/Mario_Pilot_Data/participant_csv/cleaned_outputs
